**INSTALL LIBRARIES**

In [ ]:
!pip install fastapi nest-asyncio uvicorn

CREATE CONFIG FILE


In [ ]:
yaml_config = """
              model_path: "google/vit-base-patch16-224"
              """
with open("config.yaml", "w", encoding ="utf-8") as f:
  f.write(yaml_config)

In [ ]:
!pip install transformers pillow requests torch

In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import requests

url = 'https://sieupet.com/sites/default/files/pictures/images/cho-pug-bieu-cam.jpg'
image = Image.open(requests.get(url, stream=True).raw)

processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

inputs = processor(images=image, return_tensors="pt")
outputs = model(**inputs)
logits = outputs.logits

predicted_class_idx = logits.argmax(-1).item()
print("Predicted class:", model.config.id2label[predicted_class_idx])


BUILD MODEL


In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import requests
from omegaconf import OmegaConf
import torch

class PetClassification:
  def __init__(self, config_path):
    self.config = OmegaConf.load(config_path)
    self.model = ViTForImageClassification.from_pretrained(self.config.model_path)
    self.processor = ViTImageProcessor.from_pretrained(self.config.model_path)
  def __call__(self, image: Image.Image):
        """Thực hiện suy luận và trả về nhãn dự đoán"""
        inputs = self.processor(images=image, return_tensors="pt")
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits

        predicted_class_idx = logits.argmax(-1).item()
        return self.model.config.id2label[predicted_class_idx]



In [ ]:
import requests
from PIL import Image
import io

pet_classifier = PetClassification("config.yaml")

# 1. Lấy ảnh từ URL
url = "https://sieupet.com/sites/default/files/pictures/images/cho-pug-bieu-cam.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# 2. Test model bằng cách gọi biến pet_classifier như một hàm
result = pet_classifier(image)

print(f"Model dự đoán đây là: {result}")

# Initialize API

In [ ]:
import io
import torch
import uvicorn
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import JSONResponse
from PIL import Image
from transformers import ViTImageProcessor, ViTForImageClassification
from fastapi.middleware.cors import CORSMiddleware


app = FastAPI()

# Khởi tạo mô hình
pet_classifier = PetClassification("config.yaml")

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

## TODO
@app.get('/classify')
async def intro():
    return {
        "system": "Hệ thống nhận diện thú cưng (Chó/Mèo)"
    }

@app.get("/health")
async def health():
    return {"status": "active",
            "model": "google/vit-base-patch16-224"
            }

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    """Nhận ảnh, xử lý và trả về kết quả JSON"""

    # 1. Kiểm tra định dạng đầu vào (Cơ bản)
    if not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="Đầu vào là 1 hình ảnh!!")

    try:
        # 2. Đọc dữ liệu ảnh từ buffer
        content = await file.read()
        if not content:
            raise HTTPException(status_code=400, detail="Dữ liệu ảnh bị trống.")

        image = Image.open(io.BytesIO(content)).convert("RGB")

        # 3. Gọi model từ Class để dự đoán
        label = pet_classifier(image)

        # 4. Trả về kết quả JSON rõ ràng
        return {
            "status": "success",
            "filename": file.filename,
            "prediction": label,
            "message": f"Hệ thống xác nhận đây là: {label}"
        }

    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"status": "error", "detail": f"Có lỗi xảy ra: {str(e)}"}
        )
## END TODO

import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server started on http://127.0.0.1:8000")

In [ ]:
!pip install requests

In [ ]:
import requests

API_URL = "http://127.0.0.1:8000/classify"

response = requests.get(API_URL)

response.json()

In [ ]:
import requests

API_URL = "http://127.0.0.1:8000/health"

response = requests.get(API_URL)

response.json()

In [ ]:
import requests
import io

API_URL = "http://127.0.0.1:8000/predict"

img_url = "https://sieupet.com/sites/default/files/pictures/images/cho-pug-bieu-cam.jpg"

img_data = requests.get(img_url).content

files = {'file': ('pug.jpg', io.BytesIO(img_data), 'image/jpeg')}

response = requests.post(API_URL, files=files)

print(response.json())

In [ ]:
# TODO: Chạy lệnh dưới để tunnel ra bên ngoài CHÚ Ý: NHẤN PHẢI RỒI CHỌN COPY TRONG MENU, NẾU BẤM CTRL + C SẼ TẮT KẾT NỐI
# ssh -p 443 -R0:localhost:8000 qr@a.pinggy.io

In [ ]:
import requests
import io

API_URL = "http://xfmlf-34-12-239-76.a.free.pinggy.link/predict"

img_url = "https://sieupet.com/sites/default/files/pictures/images/cho-pug-bieu-cam.jpg"

img_data = requests.get(img_url).content

files = {'file': ('pug.jpg', io.BytesIO(img_data), 'image/jpeg')}

response = requests.post(API_URL, files=files)

print(response.json())